# Detectron2 Write Models 编写模型完整 Demo

文档来源：Detectron2‑0.6 Documentation → Write Models

覆盖内容：
1. Register New Components 注册新组件（自定义 Backbone 示例）
2. 使用注册后的自定义骨干网络构建完整模型
3. Construct Models with Explicit Arguments 显式参数构造模型
4. 自定义 FastRCNNOutputLayers 损失层、自定义 ROIHeads 注册示例

## 环境检测

运行前先检查本机的 Python / PyTorch / CUDA / detectron2 版本，确认环境匹配（CUDA 与 PyTorch、detectron2 wheel 需对应）。

In [ ]:
import sys
print("Python:       ", sys.version.split()[0])

try:
    import torch
    print("PyTorch:      ", torch.__version__)
    print("CUDA (torch): ", torch.version.cuda)
    print("GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch:       未安装")

try:
    import detectron2
    print("detectron2:   ", detectron2.__version__)
except ImportError:
    print("detectron2:    未安装")

## 0. 环境准备

【Colab 取消注释执行】安装 detectron2，注意匹配 cuda、torch 版本。以下命令先注释保留，确认上一节环境检测无误后再按需执行。

In [ ]:
# Colab 取消注释执行安装
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html

import torch
import torch.nn as nn
from detectron2.config import get_cfg
from detectron2.modeling import (
    build_model,
    BACKBONE_REGISTRY, Backbone, ShapeSpec,
    ROI_HEADS_REGISTRY, StandardROIHeads,
    FastRCNNOutputLayers
)

## 1. Register New Components：注册新组件

实现自定义 Backbone，继承 `Backbone` 基类，使用注册表装饰器 `@BACKBONE_REGISTRY.register()` 注册。注册后可以直接在 cfg 配置中通过名字（字符串）使用该组件。

In [ ]:
@BACKBONE_REGISTRY.register()
class ToyBackbone(Backbone):
    """
    文档示例：极简自定义骨干网络
    输入: [B, 3, H, W] 图像tensor
    输出: 多尺度特征字典 {特征名: tensor}
    """
    def __init__(self, cfg, input_shape):
        super().__init__()
        # 简单卷积层作为骨干，stride=16，下采样16倍
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=16, padding=3)

    def forward(self, image):
        """前向传播，返回特征字典"""
        feat = self.conv1(image)
        return {"conv1": feat}

    def output_shape(self):
        """
        返回每一个输出特征的ShapeSpec元信息
        channels：通道数，stride：相对于原图的下采样倍数
        """
        return {"conv1": ShapeSpec(channels=64, stride=16)}

### 使用注册好的 ToyBackbone 构建模型

修改配置中 `BACKBONE.NAME` 为注册的类名字符串，`build_model` 自动查找注册表构建。

根源：Faster‑RCNN‑FPN 的 yaml 配置里，RPN 需要特征 `["p2","p3","p4","p5","p6"]`，但是你的 backbone 输出字典里没有 `p2` 这个 key，程序去字典取`input_shape["p2"]`直接报 KeyErrorGitHub。

In [ ]:
import torch
import torch.nn as nn
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.modeling import BACKBONE_REGISTRY, Backbone, ShapeSpec, build_model


@BACKBONE_REGISTRY.register()
class ToyBackbone(Backbone):
    def __init__(self, cfg, input_shape):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=16, padding=3)

    def forward(self, x):
        # !!!!!!!!! x 是 tensor [N,3,H,W]，不是字典！！！
        return {"conv1": self.conv1(x)}

    def output_shape(self):
        return {"conv1": ShapeSpec(channels=64, stride=16)}


cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO‑Detection/faster_rcnn_R_50_FPN_1x.yaml"))

cfg.MODEL.DEVICE = "cpu"

cfg.MODEL.RPN.IN_FEATURES = ["conv1"]
cfg.MODEL.ROI_HEADS.IN_FEATURES = ["conv1"]

cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
cfg.MODEL.ANCHOR_GENERATOR.STRIDES = [16]

cfg.MODEL.BACKBONE.NAME = "ToyBackbone"
cfg.MODEL.FPN.ON = False

model = build_model(cfg)
print(f"✅模型构建完成，backbone类型：{type(model.backbone)}")

# ----------------注意！！单独测试backbone，不能传dict！直接传tensor！----------------
dummy_img = torch.randn(2, 3, 256, 256)
features = model.backbone(dummy_img)
print(f"骨干输出特征字典keys: {list(features.keys())}")
print(f"conv1特征shape: {features['conv1'].shape}")


# --------完整模型前向loss测试（GeneralizedRCNN，输入是batched_inputs字典列表）--------
from detectron2.structures import Boxes, Instances

h,w = 256,256
gt_inst = Instances((h,w))
gt_inst.gt_boxes = Boxes(torch.tensor([[10,10,100,100]]))
gt_inst.gt_classes = torch.tensor([0])

inputs = [{"image": torch.randn(3, h, w), "instances": gt_inst}]
loss_dict = model(inputs)
print("loss_dict:", loss_dict)


In [ ]:
import os
print(os.getcwd())



> ToyBackbone 只是**注册器语法片段**，不是拿来直接`build_model`跑 Faster‑RCNN 的完整样例。
> 文档只展示怎么注册一个 Backbone，完全没有配套 RPN、Anchor、FPN 关闭后的全套配置，强行套进 Faster‑RCNN 就会连环抛错。

### 核心真相

1. `Backbone.forward(x)`接收**Tensor**，不是字典；字典是外层`GeneralizedRCNN`处理的。
2. FPN 预训练 yaml 关掉 FPN 之后，`ANCHOR_GENERATOR`、`RPN.IN_FEATURES`、`ROI_HEADS.IN_FEATURES`全部要手动对齐单特征输出。
3. detectron2 的组件耦合很强，只换 backbone，其他头部配置不匹配就疯狂断言、索引报错。

如果你只是想学会**自定义 backbone 的注册写法**，不用硬跑通完整 Faster‑RCNN 训练前向，只做骨干本身的实例化 + 前向就足够理解注册器，不需要牵扯 RPN、ROI 这些乱七八糟的组件。

In [1]:
import torch
import torch.nn as nn
from detectron2.modeling import BACKBONE_REGISTRY, Backbone, ShapeSpec

@BACKBONE_REGISTRY.register()
class ToyBackbone(Backbone):
    def __init__(self, cfg, input_shape):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=16, padding=3)

    def forward(self, x):
        return {"conv1": self.conv1(x)}

    def output_shape(self):
        return {"conv1": ShapeSpec(channels=64, stride=16)}

# 只实例化骨干，不去套Faster‑RCNN整套meta_arch
cfg_dummy = None
bb = ToyBackbone(cfg_dummy, input_shape=None)
dummy = torch.randn(1,3,256,256)
feat = bb(dummy)
print(feat["conv1"].shape)


torch.Size([1, 64, 16, 16])


## 2. Construct Models with Explicit Arguments 显式参数构造模型

场景：替换 `box_head` 的预测层，实现自定义 loss。
1. 继承 `FastRCNNOutputLayers` 重写损失逻辑
2. 使用显式参数传给 `StandardROIHeads`
3. 可选：注册自定义 ROIHeads，支持配置文件直接调用

In [ ]:
class MyRCNNOutput(FastRCNNOutputLayers):
    """
    自定义box预测层，继承FastRCNNOutputLayers，可改写loss、预测逻辑
    这里仅做占位示例，你可以改写 losses() 方法实现自定义损失
    """
    def __init__(self, cfg, input_shape):
        super().__init__(cfg, input_shape)

    def losses(self, predictions, proposals):
        """
        重写损失计算，这里可以替换为你自己的loss函数
        predictions: 网络输出的分类、回归预测
        proposals: 带gt的proposal实例
        """
        # 调用父类原始loss，实际使用替换为自定义loss逻辑
        loss_dict = super().losses(predictions, proposals)
        # 示例：对loss做修改
        for k in loss_dict:
            loss_dict[k] = loss_dict[k] * 1.0
        return loss_dict


# 2.1 手动构造 StandardROIHeads，传入自定义 box_predictor
# 获取骨干输出的特征形状信息
input_shape_dict = model.backbone.output_shape()

roi_heads = StandardROIHeads(
    cfg,
    input_shape_dict,
    box_predictor=MyRCNNOutput(cfg, input_shape_dict)
)
print(f"✅手动构造roi_heads，box_predictor类型：{type(roi_heads.box_predictor)}")

### 可选：注册自定义 ROIHeads，配置文件可以直接使用

In [ ]:
@ROI_HEADS_REGISTRY.register()
class MyStandardROIHeads(StandardROIHeads):
    def __init__(self, cfg, input_shape):
        super().__init__(
            cfg,
            input_shape,
            box_predictor=MyRCNNOutput(cfg, input_shape)
        )

# 使用示例：配置中指定 MODEL.ROI_HEADS.NAME = "MyStandardROIHeads"
cfg.MODEL.ROI_HEADS.NAME = "MyStandardROIHeads"
# 重新 build_model，会从 ROI_HEADS_REGISTRY 加载我们自定义的 ROIHeads
model2 = build_model(cfg)
print(f"✅注册后的模型 roi_heads类型：{type(model2.roi_heads)}")

# 关键知识点总结

1. Registry 注册表：把类名字符串映射到实现类，实现配置文件驱动组件替换
2. Backbone 必须实现 `forward()` 和 `output_shape()` 接口
3. 深度自定义：可以不依赖配置注册表，直接代码层面传参构造组件
4. ROIHeads、Backbone、ProposalGenerator 等都有对应的 REGISTRY